# Flan-T5 Fine-Tuning on SAMSum — Training Demo

**Runtime:** Google Colab T4 GPU  
**Goal:** Fine-tune `google/flan-t5-base` on the SAMSum dialogue summarization dataset,
evaluate with ROUGE, then push the model to Hugging Face Hub.

Steps:
1. Install dependencies
2. Load & tokenize dataset
3. Baseline ROUGE evaluation
4. Fine-tune with `Seq2SeqTrainer`
5. Post-training ROUGE comparison
6. Push to Hugging Face Hub

In [ ]:
# Cell 1 — Install dependencies (Colab uses pip, not uv)
!pip install -q \
    "torch>=2.10.0" \
    "transformers>=5.2.0" \
    "datasets>=4.6.0" \
    "evaluate>=0.4.6" \
    "rouge_score>=0.1.2" \
    "accelerate>=1.12.0" \
    "huggingface_hub>=1.4.1"

print('Installation complete.')

In [ ]:
# Cell 2 — Load and tokenize the SAMSum dataset
from datasets import load_dataset
from transformers import AutoTokenizer

MODEL_ID = "google/flan-t5-base"
INPUT_MAX_LENGTH = 512
LABEL_MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
dataset = load_dataset("samsum")

def tokenize(examples):
    inputs = ["summarize: " + d for d in examples["dialogue"]]
    model_inputs = tokenizer(inputs, max_length=INPUT_MAX_LENGTH,
                             truncation=True, padding="max_length")
    labels = tokenizer(text_target=examples["summary"],
                       max_length=LABEL_MAX_LENGTH,
                       truncation=True, padding="max_length")
    # Mask padding in labels
    model_inputs["labels"] = [
        [(t if t != tokenizer.pad_token_id else -100) for t in ids]
        for ids in labels["input_ids"]
    ]
    return model_inputs

tokenized = dataset.map(tokenize, batched=True,
                        remove_columns=dataset["train"].column_names)

print(f"Train: {len(tokenized['train'])} | Val: {len(tokenized['validation'])} | Test: {len(tokenized['test'])}")

# Smoke-check: verify 1,000 samples tokenize cleanly
sample = dataset["train"].select(range(1000))
tok_sample = sample.map(tokenize, batched=True, remove_columns=sample.column_names)
assert len(tok_sample) == 1000, "Tokenization smoke test failed"
print("Smoke test passed: 1,000 samples tokenized without error.")

In [ ]:
# Cell 3 — Baseline ROUGE evaluation (untrained model)
# transformers 5.x removed pipeline("summarization") and pipeline("text2text-generation").
# Use AutoModelForSeq2SeqLM + AutoTokenizer directly instead.
import torch
import evaluate
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

rouge = evaluate.load("rouge")

baseline_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
baseline_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID)
baseline_model.eval()

test_samples = dataset["test"].select(range(100))

def generate_summary(tokenizer, model, text, max_new_tokens=LABEL_MAX_LENGTH):
    inputs = tokenizer("summarize: " + text, return_tensors="pt",
                       truncation=True, max_length=INPUT_MAX_LENGTH)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

baseline_preds = [generate_summary(baseline_tokenizer, baseline_model, s["dialogue"])
                  for s in test_samples]
baseline_refs = test_samples["summary"]
baseline_scores = rouge.compute(predictions=baseline_preds, references=baseline_refs)

print("=== Baseline ROUGE (untrained) ===")
for k, v in baseline_scores.items():
    print(f"  {k}: {v*100:.2f}")

In [ ]:
# Cell 4 — Fine-tune with Seq2SeqTrainer (transformers 5.x API)
from transformers import (
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID)
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir="/content/flan-t5-samsum",
    eval_strategy="epoch",           # not evaluation_strategy (deprecated)
    learning_rate=5e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    save_strategy="epoch",
    load_best_model_at_end=True,
    predict_with_generate=True,
    fp16=True,
    logging_steps=100,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,      # transformers 5.x: replaces tokenizer=
    data_collator=data_collator,
)

trainer.train()
trainer.save_model("/content/flan-t5-samsum")
tokenizer.save_pretrained("/content/flan-t5-samsum")
print("Fine-tuning complete. Model saved to /content/flan-t5-samsum")

In [ ]:
# Cell 5 — Post-training ROUGE comparison
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

finetuned_tokenizer = AutoTokenizer.from_pretrained("/content/flan-t5-samsum")
finetuned_model = AutoModelForSeq2SeqLM.from_pretrained("/content/flan-t5-samsum")
finetuned_model.eval()

finetuned_preds = [generate_summary(finetuned_tokenizer, finetuned_model, s["dialogue"])
                   for s in test_samples]
finetuned_scores = rouge.compute(predictions=finetuned_preds, references=baseline_refs)

print("=== Post-Training ROUGE ===")
for k, v in finetuned_scores.items():
    print(f"  {k}: {v*100:.2f}")

print("\n=== Improvement ===")
for k in baseline_scores:
    delta = (finetuned_scores[k] - baseline_scores[k]) * 100
    print(f"  {k}: {delta:+.2f}")

In [ ]:
# Cell 6 — Push model and tokenizer to Hugging Face Hub
import os
from huggingface_hub import login
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# Set your HF token and desired repo name
HF_TOKEN  = "hf_YOUR_TOKEN_HERE"   # or use: from google.colab import userdata; HF_TOKEN = userdata.get('HF_TOKEN')
REPO_ID   = "your-username/flan-t5-samsum-summarizer"

login(token=HF_TOKEN)

model     = AutoModelForSeq2SeqLM.from_pretrained("/content/flan-t5-samsum")
tokenizer = AutoTokenizer.from_pretrained("/content/flan-t5-samsum")

model.push_to_hub(REPO_ID)
tokenizer.push_to_hub(REPO_ID)

print(f"Model live at https://huggingface.co/{REPO_ID}")